# Proyecto Integrador Big Data · 1ra Entrega Parcial

**Caso de uso:** Cloud Provider Analytics

**Objetivo.**

La consigna del proyecto solicita para esta primera entrega, como uno de los alcances obligatorios, una evidencia mínima de lectura, generación o exploración de datos.

Así pues, el presente notenook tiene como objetivo mostrar dicha evidencia respecto a las 8 fuentes de datos provistas, poniendo especial atención en métricas como claves, nulos, tipos de datos ambiguos, esquemas diferentes, integridad referencial y consistencia temporal. Esto, en pro de fungir como la justificación de las decisiones pertinentes al diseño.

Ante esto, se vuelve importante remarcar que las operaciones sobre la zona de Landing son estrictamente de zona lectura. Si bien al final se encuentra una operación de escritura, se trata del resumen de las métricas que se almacena en el directorio `evidence/entrega1`

El código puede ser ejecutado bajo dos entornos.

- *Google Colab:* Para ello, el repositorio completo debe encontrarse en `MyDrive/cloud-provider-analytics/`

- *Entorno local:* El dataset debe residir en el directorio `datalake/landing/`.

A continuación, se inicializa el entorno, detectando si la ejecución ocurre en el entorno local o en Collab, caso para el cuál monta Google Drive.
Instala además PySpark y crea el directorio para las evidencias.

El resultado confirma que se tiene acceso a `datalake/landing`

In [ ]:

import os
import sys
import json
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    project_root = os.environ.get("PROJECT_ROOT", "/content/drive/MyDrive/TP-BigData-CloudProviderAnalytics")
    os.chdir(project_root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)


REPO = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
if not (REPO / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from src.common.config import LANDING, EVENTS_DIR, EVIDENCE_DIR
from src.common.schemas import EVENTS_RAW_SCHEMA, CSV_COLUMNS, NATURAL_KEYS, all_string_schema
from src.common.spark import get_spark

OUT = EVIDENCE_DIR / "entrega1"
OUT.mkdir(parents=True, exist_ok=True)
print("Directorio del Landing:", LANDING, "\n¿existe?", LANDING.exists())

Más adelante, se inicia una sesión de Spark. El resultado inicialmente confirma el incicio correcto de Spark, pero de manera adicional permite registrar la versión.

In [ ]:
from pyspark.sql import functions as F
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

spark = get_spark("entrega1-exploracion")
print("Spark", spark.version)

resumen = {}

Es necesario realizar el inventario físico de la zona Landing, para tal objetivo se cuentan los archivos .csv y .jsonl, calculando a la par su tamaño.
Esto último nos interesa de manera particular para usage_events_stream. La comparación con un bloque HDFS permite vislumbrar que el problema es más de muchos archivos pequeños en lugar de un problema de volumen masivo.

In [ ]:
filas = []


for archivo in sorted(LANDING.glob("*.csv")):
    filas.append({"fuente": archivo.stem, "formato": "csv", "archivos": 1,
                  "bytes": archivo.stat().st_size})


archivos_eventos = sorted(EVENTS_DIR.glob("*.jsonl"))
bytes_eventos = 0
for archivo in archivos_eventos:
    bytes_eventos = bytes_eventos + archivo.stat().st_size
filas.append({"fuente": "usage_events_stream", "formato": "jsonl",
              "archivos": len(archivos_eventos), "bytes": bytes_eventos})

inventario = pd.DataFrame(filas)
inventario["KB"] = (inventario["bytes"] / 1024).round(1)

kb_por_archivo = bytes_eventos / 1024 / len(archivos_eventos)
veces = (128 * 1024) / kb_por_archivo
print(f"Tamaño promedio de un archivo de eventos: {kb_por_archivo:.1f} KB")
print(f"Un bloque HDFS (128 MB) es {veces:,.0f} veces más grande")

resumen["inventario_fisico"] = inventario.to_dict("records")
inventario

Se realiza una lectura de las 7 fuentes .csv utilizando un esquema explícito en el que todas las columnas de refuerzan temporalmente a StringType, se evita el uso de inferSchema para evitar la conversión a NULL de valoresa ambiguos.
Para cada fuete se calcula cantidad de filas , columnas, duplicados sobre clabve natural definida y valores nulos.
Se observó que un campo que contiene comas va entre comillas, y una comilla interna se escribe duplicada (`"[""env:prod"", ""pii:true""]"`). Spark usa por defecto la barra invertida como carácter de escape, por lo que sin `option("escape", '"')` el campo `tags_json` de `resources.csv` se corta en la primera coma interna y se pierde información.

In [ ]:
def leer_csv_crudo(nombre):
    ruta = str(LANDING / f"{nombre}.csv")
    esquema = all_string_schema(CSV_COLUMNS[nombre])
    return (spark.read
            .option("header", True)
            .option("escape", '"')
            .schema(esquema)
            .csv(ruta))


def perfil(nombre, df, clave):
    total = df.count()
    claves_distintas = df.select(clave).distinct().count()
    detalle = []
    for columna in df.columns:
        nulos = df.filter(F.col(columna).isNull()).count()
        detalle.append({"columna": columna, "nulos": nulos,
                        "pct_nulos": round(100 * nulos / total, 1)})
    return {"fuente": nombre, "filas": total, "columnas": len(df.columns),
            "clave": "+".join(clave), "duplicados_clave": total - claves_distintas,
            "detalle": detalle}



csv = {}
for nombre in CSV_COLUMNS:
    csv[nombre] = leer_csv_crudo(nombre)


perfiles = []
tabla = []
for nombre in csv:
    p = perfil(nombre, csv[nombre], NATURAL_KEYS[nombre])
    perfiles.append(p)
    columnas_con_nulos = []
    for d in p["detalle"]:
        if d["nulos"] > 0:
            columnas_con_nulos.append(f"{d['columna']} ({d['pct_nulos']}%)")
    tabla.append({"fuente": nombre, "filas": p["filas"], "columnas": p["columnas"],
                  "clave": p["clave"], "duplicados_clave": p["duplicados_clave"],
                  "columnas_con_nulos": ", ".join(columnas_con_nulos)})

resumen["perfil_csv"] = perfiles
pd.DataFrame(tabla)

Se ejecutan chequeos con respecto a las fuentes, clasificando los hallazgos acorde a la siguiente tabla

| Tipo |  | Tratamiento previsto |
|---|---|---|
| **Calidad** | Datos inválidos, incompletos o inconsistentes. Se subclasifican por dimensión: *validez*, referente a datos  que no respetan limites matemáticos o formatos permitidos; *completitud*, cuando falta información; y *consistencia*, para cuando existen contradicciones lógicas| Flag, corrección documentada o quarantine |
| **Negocio** | Datos estructuralmente perfectos, pero que revelan eventos críticos | No se toman como errores |
| **Gobierno** | Datos personales o sensibles (PII) | Clasificación, enmascaramiento y control de acceso|

In [ ]:
def como_numero(columna):
    return F.col(columna).cast("double")

def como_fecha(columna):
    return F.to_date(F.col(columna))



chequeos = [
    ("billing_monthly", "Calidad", "Consistencia", "currency = USD con exchange_rate_to_usd <> 1",
        (F.col("currency") == "USD") & (como_numero("exchange_rate_to_usd") != 1)),
    ("billing_monthly", "Negocio", "-", "subtotal negativo (semántica abierta, ver D-09)",
        como_numero("subtotal") < 0),
    ("billing_monthly", "Calidad", "Completitud", "credits nulo",
        F.col("credits").isNull()),
    ("billing_monthly", "Negocio", "-", "moneda distinta de USD (requiere normalización)",
        F.col("currency") != "USD"),
    ("customers_orgs",  "Calidad", "Completitud", "nps_score nulo",
        F.col("nps_score").isNull()),
    ("customers_orgs",  "Calidad", "Validez", "nps_score fuera de [-100, 100]",
        (como_numero("nps_score") < -100) | (como_numero("nps_score") > 100)),
    ("nps_surveys",     "Calidad", "Completitud", "nps_score nulo",
        F.col("nps_score").isNull()),
    ("nps_surveys",     "Calidad", "Validez", "nps_score fuera de [0, 10] (escala individual)",
        (como_numero("nps_score") < 0) | (como_numero("nps_score") > 10)),
    ("support_tickets", "Negocio", "-", "ticket abierto (resolved_at nulo)",
        F.col("resolved_at").isNull()),
    ("support_tickets", "Calidad", "Completitud", "csat nulo",
        F.col("csat").isNull()),
    ("support_tickets", "Calidad", "Validez", "csat fuera de [1, 5]",
        (como_numero("csat") < 1) | (como_numero("csat") > 5)),
    ("support_tickets", "Calidad", "Consistencia", "resolved_at < created_at",
        como_fecha("resolved_at") < como_fecha("created_at")),
    ("support_tickets", "Negocio", "-", "severity = critical",
        F.col("severity") == "critical"),
    ("support_tickets", "Negocio", "-", "sla_breached = True",
        F.col("sla_breached") == "True"),
    ("users",           "Calidad", "Consistencia", "last_login < created_at",
        como_fecha("last_login") < como_fecha("created_at")),
    ("users",           "Calidad", "Completitud", "last_login nulo",
        F.col("last_login").isNull()),
    ("users",           "Gobierno", "PII", "email presente (dato personal)",
        F.col("email").isNotNull()),
    ("resources",       "Gobierno", "PII", "tags_json contiene pii:true",
        F.col("tags_json").contains("pii:true")),
    ("customers_orgs",  "Calidad", "Consistencia", "is_enterprise no coincide con plan_tier = enterprise",
        (F.col("is_enterprise") == "True") != (F.col("plan_tier") == "enterprise")),
    ("marketing_touches", "Calidad", "Consistencia", "converted = True sin clicked",
        (F.col("converted") == "True") & (F.col("clicked") == "False")),
]

filas = []
for fuente, tipo, dimension, descripcion, condicion in chequeos:
    df = csv[fuente]
    casos = df.filter(condicion).count()
    total = df.count()
    filas.append({"tipo": tipo, "dimension": dimension, "fuente": fuente, "chequeo": descripcion,
                  "casos": casos, "total": total, "pct": round(100 * casos / total, 1)})

resumen["chequeos_maestros"] = filas
pd.DataFrame(filas).sort_values(["tipo", "dimension", "fuente"]).reset_index(drop=True)

Se calcula el rango temporal de las fuentes, calculando la fecha más antigua, la más reciente y la cantidad de fechas únicas; y se analizan las tasas de cambio

In [ ]:

columnas_fecha = [("billing_monthly", "month"), ("customers_orgs", "signup_date"),
                  ("support_tickets", "created_at"), ("marketing_touches", "timestamp"),
                  ("nps_surveys", "survey_date"), ("users", "created_at"),
                  ("resources", "created_at")]
rangos = []
for fuente, columna in columnas_fecha:
    fila = csv[fuente].agg(F.min(como_fecha(columna)).alias("desde"),
                           F.max(como_fecha(columna)).alias("hasta"),
                           F.countDistinct(como_fecha(columna)).alias("fechas_distintas")).first()
    rangos.append({"fuente": fuente, "columna": columna, "desde": str(fila["desde"]),
                   "hasta": str(fila["hasta"]), "fechas_distintas": fila["fechas_distintas"]})
resumen["rangos_temporales"] = rangos

fx = (csv["billing_monthly"]
      .groupBy("currency")
      .agg(F.count("*").alias("facturas"),
           F.round(F.min(como_numero("exchange_rate_to_usd")), 5).alias("fx_min"),
           F.round(F.max(como_numero("exchange_rate_to_usd")), 5).alias("fx_max")))
print(fx.toPandas())
pd.DataFrame(rangos)

Se verifica la integridad referencial de los maestros: todas las fuentes se relacionan por `org_id` con `customers_orgs`, por lo que se cuentan los huérfanos (filas cuyo `org_id` no existe en el maestro de clientes) y la cantidad de filas por organización. Además, se comprueba que la facturación cumpla el grano esperado de una factura por organización y mes, y que el email de los usuarios sea único.

In [ ]:
orgs = csv["customers_orgs"].select("org_id")
integridad_maestros = []
for fuente in ["users", "resources", "support_tickets", "marketing_touches", "nps_surveys", "billing_monthly"]:
    df = csv[fuente]
    huerfanos = df.join(orgs, on="org_id", how="left_anti").count()
    orgs_con_filas = df.select("org_id").distinct().count()
    integridad_maestros.append({"fuente": fuente, "filas": df.count(), "huerfanos_org_id": huerfanos,
                                "orgs_con_filas": orgs_con_filas,
                                "filas_por_org": round(df.count() / orgs_con_filas, 1)})

facturas = csv["billing_monthly"]
dup_org_mes = facturas.count() - facturas.select("org_id", "month").distinct().count()
emails_dup = csv["users"].count() - csv["users"].select("email").distinct().count()
print(f"Facturas duplicadas por (org_id, month): {dup_org_mes}")
print(f"Emails repetidos en users: {emails_dup}")

resumen["integridad_maestros"] = {"por_fuente": integridad_maestros,
                                  "billing_dup_org_mes": dup_org_mes, "users_email_dup": emails_dup}
pd.DataFrame(integridad_maestros)

Se realiza una lectura de `usage_events_stream` utilizando un esquema explícito que abarca en simultáneo las versiones v1 y v2 del esquema. Para este caso se repite la estrategia de leer los valores numéricos cual si fueran texto.
El modo PERMISSIVE es configurado en pro de que al encontrar una línea de JSON rota, no se detenga el proceso.

Se observó que 14 eventos comparten `(org_id, resource_id, timestamp, metric)` con otro evento, pero se trata de mediciones distintas que coincidieron en el mismo minuto.

In [ ]:
ev = (spark.read
      .schema(EVENTS_RAW_SCHEMA)
      .option("mode", "PERMISSIVE")
      .option("columnNameOfCorruptRecord", "_corrupt_record")
      .json(str(EVENTS_DIR))
      .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1))
      .withColumn("event_ts", F.to_timestamp("timestamp"))
      .withColumn("value_num", F.col("value").cast("double"))
      .withColumn("cost_num", F.col("cost_usd_increment").cast("double"))
      .cache())

total_ev = ev.count()
rango = ev.agg(F.min("event_ts").alias("desde"), F.max("event_ts").alias("hasta")).first()

basicos = {}
basicos["eventos"] = total_ev
basicos["archivos"] = ev.select("source_file").distinct().count()
basicos["registros_corruptos"] = ev.filter(F.col("_corrupt_record").isNotNull()).count()
basicos["timestamp_no_parseable"] = ev.filter(F.col("event_ts").isNull()).count()
basicos["event_id_nulo"] = ev.filter(F.col("event_id").isNull()).count()
basicos["event_id_duplicado"] = total_ev - ev.select("event_id").distinct().count()
basicos["colisiones_clave_contextual(org,resource,timestamp,metric)"] = (
    total_ev - ev.select("org_id", "resource_id", "timestamp", "metric").distinct().count())
basicos["organizaciones"] = ev.select("org_id").distinct().count()
basicos["recursos"] = ev.select("resource_id").distinct().count()
basicos["dias_con_eventos"] = ev.select(F.to_date("event_ts")).distinct().count()
basicos["desde"] = str(rango["desde"])
basicos["hasta"] = str(rango["hasta"])

resumen["eventos_basicos"] = basicos
pd.Series(basicos, name="valor").to_frame()

Se analiza la evolución del esquema de los eventos

In [ ]:
versiones = (ev.groupBy("schema_version")
             .agg(F.count("*").alias("eventos"),
                  F.min("event_ts").alias("primer_evento"),
                  F.max("event_ts").alias("ultimo_evento"),
                  F.count("carbon_kg").alias("con_carbon_kg"),
                  F.count("genai_tokens").alias("con_genai_tokens"))
             .orderBy("schema_version")
             .toPandas())

tokens_por_servicio = (ev.filter(F.col("genai_tokens").isNotNull())
                       .groupBy("service")
                       .count()
                       .toPandas())
print("Eventos con genai_tokens por servicio:")
print(tokens_por_servicio.to_string(index=False))

resumen["versiones_esquema"] = versiones.astype(str).to_dict("records")
resumen["genai_tokens_por_servicio"] = tokens_por_servicio.to_dict("records")
versiones

Se miden valores nulos y tipos anbiguos en los eventos, considerando para tal propósito aquellos `value` que llegaron como texto en el JSON crudo, aquellos que pudieron convertirse a número y los `unit` nulos.

In [ ]:
lineas = spark.read.text(str(EVENTS_DIR))
value_como_texto = lineas.filter(F.col("value").contains('"value": "')).count()
cost_como_texto = lineas.filter(F.col("value").contains('"cost_usd_increment": "')).count()

nulos_ev = {}
for columna in ["org_id", "resource_id", "service", "region", "metric", "value",
                "unit", "cost_usd_increment"]:
    nulos_ev[columna] = ev.filter(F.col(columna).isNull()).count()

tipos = {
    "value_nulo": nulos_ev["value"],
    "value_llego_como_texto": value_como_texto,
    "value_no_casteable_a_numero": ev.filter(F.col("value").isNotNull() & F.col("value_num").isNull()).count(),
    "cost_llego_como_texto": cost_como_texto,
    "unit_nulo": nulos_ev["unit"],
}
resumen["eventos_nulos"] = nulos_ev
resumen["eventos_tipos"] = tipos
print(pd.Series(tipos, name="eventos").to_string())

metric_unit = (ev.groupBy("metric")
               .pivot("unit")
               .count()
               .fillna(0)
               .orderBy("metric")
               .toPandas())
resumen["metric_unit"] = metric_unit.rename(columns={"null": "unit_nulo"}).to_dict("records")
metric_unit

Haciendo uso de percentiles y extremos, se cuentan los costos negativos y se detectan valores superiores a 10 veces el p99.

In [ ]:
p50, p90, p99, p999 = ev.approxQuantile("cost_num", [0.5, 0.9, 0.99, 0.999], 0.0001)
extremos = ev.agg(F.min("cost_num").alias("min"), F.max("cost_num").alias("max")).first()

costos = {
    "min": extremos["min"],
    "p50": round(p50, 4),
    "p90": round(p90, 4),
    "p99": round(p99, 4),
    "p99.9": round(p999, 4),
    "max": extremos["max"],
    "negativos (< 0)": ev.filter(F.col("cost_num") < 0).count(),
    "negativos bajo tolerancia (< -0.01)": ev.filter(F.col("cost_num") < -0.01).count(),
    "mayores a 10x p99": ev.filter(F.col("cost_num") > 10 * p99).count(),
}
resumen["costos"] = costos
print(pd.Series(costos, name="cost_usd_increment").to_string())

por_servicio = (ev.groupBy("service")
                .agg(F.count("*").alias("eventos"),
                     F.round(F.sum("cost_num"), 2).alias("costo_total_usd"),
                     F.round(F.percentile_approx("cost_num", 0.5), 4).alias("p50"),
                     F.round(F.percentile_approx("cost_num", 0.99), 4).alias("p99"),
                     F.round(F.max("cost_num"), 2).alias("max"),
                     F.count(F.when(F.col("cost_num") < 0, 1)).alias("negativos"))
                .orderBy(F.desc("costo_total_usd"))
                .toPandas())
resumen["costos_por_servicio"] = por_servicio.to_dict("records")
por_servicio

Se realiza un contraste de los eventos de uso contra `cutomers_orgs` y `resources` mediante el uso de joins. Una de las cuestiones que se hallaron tiene que ver con inconsistencias entre la fecha del evento y el `created_at` del recurso, por lo que se propone marcarlos como anomalía

In [ ]:
recursos = csv["resources"].select(
    "resource_id",
    F.col("org_id").alias("org_r"),
    F.col("service").alias("svc_r"),
    F.col("region").alias("reg_r"),
    F.to_date("created_at").alias("res_created"))


j = ev.join(recursos, on="resource_id", how="left")


huerfanos_org = ev.join(csv["customers_orgs"].select("org_id"), on="org_id", how="left_anti")

integridad = {
    "org_id sin maestro (customers_orgs)": huerfanos_org.count(),
    "resource_id sin maestro (resources)": j.filter(F.col("org_r").isNull()).count(),
    "org del evento <> org del recurso": j.filter(F.col("org_id") != F.col("org_r")).count(),
    "service del evento <> service del recurso": j.filter(F.col("service") != F.col("svc_r")).count(),
    "region del evento <> region del recurso": j.filter(F.col("region") != F.col("reg_r")).count(),
    "evento anterior a la creación del recurso": j.filter(F.to_date("event_ts") < F.col("res_created")).count(),
}
resumen["integridad_eventos"] = integridad
pd.Series(integridad, name="eventos").to_frame()

Se evalúa la viabilidad de procesar los ecentos mediante Structured Streaming haciendo uso de watermarks. En el código a continuación se asume un escenario de ingesta de micro-lotes procesando un archivo a la vez. El resumen estadísitico muestra que cada archivo abarca en promedio casi 60 días, por lo que los mismos no se encuentran particionados de manera cronológica.

In [ ]:

por_archivo = (ev.groupBy("source_file")
               .agg(F.min("event_ts").alias("min_ts"),
                    F.max("event_ts").alias("max_ts"),
                    F.count("*").alias("eventos"))
               .orderBy("source_file")
               .toPandas())
por_archivo["dias_cubiertos"] = (por_archivo["max_ts"] - por_archivo["min_ts"]).dt.total_seconds() / 86400
print("Días cubiertos por cada archivo:")
print(por_archivo["dias_cubiertos"].describe().round(2).to_string())


por_archivo["max_previo"] = por_archivo["max_ts"].cummax().shift(1)
previo = spark.createDataFrame(por_archivo[["source_file", "max_previo"]].dropna())


ev_con_previo = ev.join(previo, on="source_file")
tardios = {}
for horas in [1, 24, 72, 168]:
    limite = F.col("max_previo") - F.expr(f"INTERVAL {horas} HOURS")
    tardios[f"watermark {horas} h"] = ev_con_previo.filter(F.col("event_ts") < limite).count()

tabla_tardios = {}
for nombre, cantidad in tardios.items():
    tabla_tardios[nombre] = f"{cantidad} ({100 * cantidad / total_ev:.1f} %)"

resumen["orden_temporal"] = {"dias_cubiertos_por_archivo_promedio": round(por_archivo["dias_cubiertos"].mean(), 2),
                             "eventos_tardios_segun_watermark": tardios}
pd.Series(tabla_tardios, name="eventos que quedarían fuera").to_frame()

In [ ]:
eventos_por_dia = ev.groupBy(F.to_date("event_ts").alias("dia")).count()
epd = eventos_por_dia.agg(F.min("count"), F.avg("count"), F.max("count")).first()
resumen["eventos_por_dia"] = {"min": epd[0], "promedio": epd[1], "max": epd[2]}
print(f"Eventos por día: min={epd[0]}, promedio={epd[1]:.0f}, max={epd[2]}")


ev.groupBy("service").pivot("metric").count().orderBy("service").toPandas()

De cara a la normalización a USD que a posterior deberá hacerse, se mide el tamaño de los montos por moneda, la relación entre el subtotal y los impuestos, la perseverancia en el tiempo de los clientes usando una misma divisa, el revenue según tres criterios de conversión y la relación entre lo facturado y el costo de uso medido en los eventos.

In [ ]:
factura = (csv["billing_monthly"]
           .withColumn("subtotal_n", como_numero("subtotal"))
           .withColumn("credits_n", F.coalesce(como_numero("credits"), F.lit(0.0)))
           .withColumn("taxes_n", como_numero("taxes"))
           .withColumn("fx", como_numero("exchange_rate_to_usd"))
           .withColumn("total_moneda_original", F.col("subtotal_n") - F.col("credits_n") + F.col("taxes_n")))


montos_por_moneda = (factura.groupBy("currency")
                     .agg(F.count("*").alias("facturas"),
                          F.round(F.percentile_approx("subtotal_n", 0.5), 2).alias("subtotal_mediana"),
                          F.round(F.min("subtotal_n"), 2).alias("subtotal_min"),
                          F.round(F.max("subtotal_n"), 2).alias("subtotal_max"),
                          F.round(F.avg("fx"), 5).alias("fx_promedio"))
                     .orderBy("currency")
                     .toPandas())
print("(a) Subtotales por moneda")
print(montos_por_moneda.to_string(index=False))


ratio_impuestos = (factura.withColumn("ratio", F.round(F.col("taxes_n") / F.col("subtotal_n"), 3))
                   .groupBy("ratio").count().orderBy(F.desc("count")).toPandas())
print("\n(b) taxes / subtotal")
print(ratio_impuestos.to_string(index=False))


monedas_por_org = factura.groupBy("org_id").agg(F.countDistinct("currency").alias("monedas"))
orgs_multimoneda = monedas_por_org.filter(F.col("monedas") > 1).count()
print(f"\n(c) Organizaciones que cambian de moneda entre meses: {orgs_multimoneda} de {monedas_por_org.count()}")

In [ ]:

fx_usd_forzado = F.when(F.col("currency") == "USD", F.lit(1.0)).otherwise(F.col("fx"))
revenue_opciones = (factura
                    .withColumn("A_tasa_tal_cual", F.col("total_moneda_original") * F.col("fx"))
                    .withColumn("B_usd_forzado_a_1", F.col("total_moneda_original") * fx_usd_forzado)
                    .withColumn("C_sin_aplicar_tasa", F.col("total_moneda_original"))
                    .groupBy("month")
                    .agg(F.round(F.sum("A_tasa_tal_cual"), 0).alias("A_tasa_tal_cual"),
                         F.round(F.sum("B_usd_forzado_a_1"), 0).alias("B_usd_forzado_a_1"),
                         F.round(F.sum("C_sin_aplicar_tasa"), 0).alias("C_sin_aplicar_tasa"))
                    .orderBy("month")
                    .toPandas())
totales = revenue_opciones[["A_tasa_tal_cual", "B_usd_forzado_a_1", "C_sin_aplicar_tasa"]].sum()
diferencia_pct = 100 * (totales["C_sin_aplicar_tasa"] - totales["A_tasa_tal_cual"]) / totales["C_sin_aplicar_tasa"]
print("(d) Revenue por mes según criterio (USD)")
print(revenue_opciones.to_string(index=False))
print(f"Totales: A={totales['A_tasa_tal_cual']:,.0f} | B={totales['B_usd_forzado_a_1']:,.0f} | "
      f"C={totales['C_sin_aplicar_tasa']:,.0f}  -> A es {diferencia_pct:.1f} % menor que C")

ars = factura.filter(F.col("currency") == "ARS").agg(
    F.round(F.sum("total_moneda_original"), 0).alias("sin_tasa"),
    F.round(F.sum(F.col("total_moneda_original") * F.col("fx")), 0).alias("con_tasa")).first()
print(f"Facturas en ARS: {ars['sin_tasa']:,.0f} sin aplicar tasa -> {ars['con_tasa']:,.0f} USD aplicándola")

In [ ]:
uso_mensual = (ev.withColumn("month", F.date_format(F.trunc("event_ts", "month"), "yyyy-MM-dd"))
               .groupBy("org_id", "month")
               .agg(F.sum("cost_num").alias("costo_uso_usd")))
relacion = (factura.join(uso_mensual, on=["org_id", "month"], how="inner")   # junio no tiene eventos
            .withColumn("subtotal_sobre_uso", F.col("subtotal_n") / F.col("costo_uso_usd"))
            .groupBy("month")
            .pivot("currency")
            .agg(F.round(F.percentile_approx("subtotal_sobre_uso", 0.5), 2))
            .orderBy("month")
            .toPandas())
print("(e) Mediana de subtotal / costo de uso, por mes y moneda (junio no tiene eventos)")
print(relacion.to_string(index=False))

resumen["facturacion_moneda"] = {
    "montos_por_moneda": montos_por_moneda.to_dict("records"),
    "ratio_impuestos_subtotal": ratio_impuestos.to_dict("records"),
    "orgs_que_cambian_de_moneda": orgs_multimoneda,
    "revenue_por_mes_segun_criterio": revenue_opciones.to_dict("records"),
    "revenue_total_segun_criterio": {k: float(v) for k, v in totales.items()},
    "ars_sin_tasa_vs_con_tasa": {"sin_tasa": ars["sin_tasa"], "con_tasa": ars["con_tasa"]},
    "mediana_subtotal_sobre_uso": relacion.to_dict("records"),
}

La siguiente tabla es generada a partir de las métricas calculadas arriba.

In [ ]:
ev_inv = resumen["inventario_fisico"][-1]
chq = {f["chequeo"]: f for f in resumen["chequeos_maestros"]}
fm = resumen["facturacion_moneda"]["revenue_total_segun_criterio"]
dif_fx = 100 * (fm["C_sin_aplicar_tasa"] - fm["A_tasa_tal_cual"]) / fm["C_sin_aplicar_tasa"]
tard = resumen["orden_temporal"]["eventos_tardios_segun_watermark"]["watermark 24 h"]

hallazgos = [
    ("120 archivos JSONL pequeños",
     f"{ev_inv['KB'] / ev_inv['archivos']:.1f} KB por archivo", "Compactación al promover a Bronze; particionado grueso (small files)"),
    ("Volumen bajo en la muestra",
     f"{resumen['eventos_basicos']['eventos']:,} eventos · {ev_inv['KB'] / 1024:.1f} MB · 720 por día",
     "Big Data se justifica por velocidad, variedad y veracidad, no por volumen"),
    ("Evolución de esquema v1 → v2",
     "corte limpio el 2025-07-18", "Esquema superset; compatibilización v1/v2 en Silver (NULL ≠ 0)"),
    ("value NULL",
     f"{resumen['eventos_tipos']['value_nulo']} eventos", "Regla de calidad (completitud)"),
    ("value como texto",
     f"{resumen['eventos_tipos']['value_llego_como_texto']} eventos, todos casteables", "Lectura como texto + cast controlado (D-05)"),
    ("unit NULL",
     f"{resumen['eventos_tipos']['unit_nulo']} eventos; metric→unit fija", "Imputación de unit por métrica + flag"),
    ("Costos negativos",
     f"{resumen['costos']['negativos (< 0)']} (< 0) · {resumen['costos']['negativos bajo tolerancia (< -0.01)']} (< −0,01)",
     "Flag de anomalía con tolerancia −0,01 (no quarantine)"),
    ("Spikes de costo",
     f"{resumen['costos']['mayores a 10x p99']} eventos > 10 × p99; escalas distintas por servicio",
     "Anomalías por servicio con percentiles/MAD"),
    ("event_id único",
     f"{resumen['eventos_basicos']['event_id_duplicado']} duplicados", "Clave de deduplicación = event_id"),
    ("Colisiones de clave contextual",
     f"{resumen['eventos_basicos']['colisiones_clave_contextual(org,resource,timestamp,metric)']} eventos, no son duplicados",
     "No usar (org, resource, timestamp, metric) como clave"),
    ("Eventos anteriores a la creación del recurso",
     f"{resumen['integridad_eventos']['evento anterior a la creación del recurso']} eventos",
     "Regla de consistencia temporal: marcar, no eliminar"),
    ("Archivos con eventos de todo el período",
     f"{tard:,} eventos tardíos con watermark 24 h (1 archivo por micro-lote)",
     "Captura sin filtro temporal; dedupe/publicación con control de estado a Bronze + agregados diarios en batch (D-04)"),
    ("Moneda de facturación inconsistente",
     f"revenue varía {dif_fx:.1f} % según el criterio de conversión", "Normalización FX como decisión abierta (D-08)"),
    ("Tickets abiertos / críticos / SLA",
     f"{chq['ticket abierto (resolved_at nulo)']['casos']} abiertos · {chq['severity = critical']['casos']} críticos · "
     f"{chq['sla_breached = True']['casos']} SLA breach", "Condiciones de negocio: insumo del mart de Soporte"),
    ("PII en users y resources",
     f"{chq['email presente (dato personal)']['casos']} emails · {chq['tags_json contiene pii:true']['casos']} recursos pii:true",
     "Clasificación, enmascaramiento y control de acceso (gobierno)"),
]
tabla_hallazgos = pd.DataFrame(hallazgos, columns=["hallazgo", "evidencia", "impacto en el diseño"])
resumen["hallazgos"] = tabla_hallazgos.to_dict("records")
tabla_hallazgos

Cada decisión del diseño se desprende de un hallazgo medido en este notebook:

| Hallazgo | Decisión a tomar | Se propone aplicar en: |
|---|---|---|
| Eventos fragmentados en 120 archivos JSONL | Ingesta con Structured Streaming, esquema explícito y checkpointing | Ingesta desde el Landing y escritura en Bronze |
| Cada archivo abarca un promedio de casi 60d | Captura sin filtro temporal; dedupe/publicación con control de estado | Streaming para ingestar y batch para procesar históricamente |
| Los archivos maestros son .cvs pequeños con baja frecuencia de actualización | Ingesta batch programada | Ingesta hacia la capa Bronze |
| 120 archivos con tamaño promedio: 105 KB | Compactar los archivos a Parquet, creando carpetas separadas por fecha | Diseño del Data Lake |
| Tipos de datos ambiguos. Ej. value como texto | Leer como texto y luego frozar conversión, si falla, se envía a quarantine | Estandarización en Bronze y carpeta quarantine |
| unit nulo | Imputación documentada + columna booleana flag | Silver |
| Esquemas diferentes v1/v2 | Esquema universal con columnas de v1 y v2. NULL en campos que no existían | Silver |
| Costos negativos y spikes por servicio | Flags de anomalía caculado mantemáticamente | Silver. Resultados entregados en Data Mart de FinOps|
| event_id único sin duplicados, pero cruzar organización, recurso, fecha y métrica arrojó 14 duplicados | Se usa el event_id para garantizar idempotencia | Deduplicación segura en Bronze |
| Eventos previos a la creación del recurso | Regla de consistencia temporal con flag | Silver |
| Moneda de facturación inconsistente | Conservar monto, moneda y tasa | Silver y Data Mart de FinOps |
| Se tienen datos clasificados como información personal identificable | Clasificarcolumnas, aplicar políticas de enmascaramiento y restringir acceso | Política de Gobierno del Dato |

Se realiza un guardado de la información acumulada en `resumen` en `evidence/entrega1/perfil_fuentes.json`

In [ ]:
from datetime import datetime, timezone
resumen["generado_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
resumen["spark_version"] = spark.version

ruta_salida = OUT / "perfil_fuentes.json"
with open(ruta_salida, "w", encoding="utf-8") as archivo:
    json.dump(resumen, archivo, ensure_ascii=False, indent=2, default=str)
print("Guardado:", ruta_salida.relative_to(REPO))